# 11a — Lay of the land: the step-10 deltas

The second interlude (convention: 020 §The book). NOT a restatement of the
ch07a baseline — only what CHANGED. The base pack grew from 10 to 15 syntax
forms; run the ch07a refusal kernels and watch three of the six graduate:

In [1]:
import pdum.dsl  # noqa: F401
from pdum.dsl.kernel import types as T
from pdum.dsl.kernel.api import jit
from pdum.dsl.kernel.lower import lower_handle
from pdum.dsl.kernel.ops import CORE_OPS
from pdum.dsl.kernel.registry import DEFAULT
from pdum.dsl.stdlib.base_lang import LOWER_RULES

print("the base pack now accepts", len([k for k in LOWER_RULES if isinstance(k, type)]), "syntax forms:")
for node_type in LOWER_RULES:
    if isinstance(node_type, type):
        print("  ✅ ast." + node_type.__name__)

the base pack now accepts 21 syntax forms:
  ✅ ast.Expr
  ✅ ast.Assign
  ✅ ast.AugAssign
  ✅ ast.Return
  ✅ ast.If
  ✅ ast.For
  ✅ ast.While
  ✅ ast.Break
  ✅ ast.Continue
  ✅ ast.Pass
  ✅ ast.Constant
  ✅ ast.Name
  ✅ ast.BinOp
  ✅ ast.UnaryOp
  ✅ ast.BoolOp
  ✅ ast.Compare
  ✅ ast.IfExp
  ✅ ast.Tuple
  ✅ ast.Subscript
  ✅ ast.Attribute
  ✅ ast.Call


In [2]:
# ch07a's refusal kernels, re-run against today's pack:
@jit()
def wants_tuple(x):
    return (x, x)


@jit()
def wants_boolop(x):
    return x > 0.0 and x < 1.0


@jit()
def wants_augassign(x):
    x += 1.0
    return x


@jit()
def wants_for(x):
    total = x
    for i in range(4):
        total = total + x
    return total


rules = dict(LOWER_RULES)
for h in (wants_tuple, wants_boolop, wants_augassign, wants_for):
    try:
        lower_handle(h, rules, {**CORE_OPS, **DEFAULT.ops}, arg_types=(T.f64,), context={"registry": DEFAULT})
        print(f"  ✅ {h.fntype.template.label}: lowers now (step 10)")
    except Exception as e:
        print(f"  🏗️ {type(e).__name__}: {e}")

  ✅ wants_tuple: lowers now (step 10)
  ✅ wants_boolop: lowers now (step 10)
  ✅ wants_augassign: lowers now (step 10)
  ✅ wants_for: lowers now (step 10)


## The delta table

| construct | ch07a | now |
|---|---|---|
| aug-assign `x += 1` | 🔧 | ✅ base pack |
| `and`/`or` | 🔧 + policy | ✅ short-circuit via `core.if` (policy: strict bool operands; truthiness stays a dialect choice) |
| chained `0 < x < 1` | 🔧 | ✅ operand evaluated once (shared node) |
| tuples + unpacking + `t[0]` | 🔧 | ✅ `core.tuple`; folds away on targets that cannot spell it |
| `.field` attribute | 🔧 pending records | ✅ `@record` dataclasses + inlined methods |
| calls beyond casts/captures | 🚫 | ✅ overloads: intrinsics + DSL batteries (surface B) |
| `if`/`for` statements, early return, `while`, `a[i]` on arrays | 🏗️ | 🏗️ step 11, unchanged |

Still true from ch07a: single tail return (settled), no NaN-as-data, loud
refusals everywhere else. The next delta lands with the arrays step.